# Reinhardt Blog Demo — Manim scenes

Renders Scenes **1**, **2**, and **10** of the 15-minute blog demo (see `reinhardt/.ignore/conte.md`).

- Scene 1 — Opening / problem framing (0:00–1:30)
- Scene 2 — Reinhardt intro + polylithic architecture (1:30–2:30)
- Scene 10 — Recap, comparison, CTA (13:30–15:00)

Use `-ql` for fast iteration and `-qh` for final render. Output MP4 files land under `media/videos/manim/`.

Theme colors follow the conte memo: background `#1a1b26` (Tokyo Night), Reinhardt accent `#f74c00`.

In [1]:
%load_ext manim

from manim import *

config.background_color = "#1a1b26"

BG        = "#1a1b26"
FG        = "#c0caf5"
ACCENT    = "#f74c00"   # Reinhardt brand orange
RUST      = "#ce422b"
DJANGO    = "#0c4b33"
MUTED     = "#565f89"
OK        = "#9ece6a"
WARN      = "#e0af68"

# Install on macOS if missing: `brew install --cask font-inter`
MONO_FONT = "Inter"
Text.set_default(font=MONO_FONT, color=FG)

The manim module is not an IPython extension.


## Scene 1 — Opening / problem framing (1:30)

In [2]:
%%manim -ql -v WARNING Scene1Opening

import numpy as np

class Scene1Opening(Scene):
    def construct(self):
        # 0:00–0:15  "Rust is fast." + speed bars
        headline = Text("Rust is fast.", font_size=72, weight=BOLD, color=FG)
        self.play(FadeIn(headline, shift=UP * 0.5), run_time=1.0)
        self.wait(0.5)
        self.play(headline.animate.to_edge(UP), run_time=0.6)

        langs  = ["Rust", "Go", "Python", "Ruby"]
        speeds = [1.00, 0.72, 0.18, 0.12]
        colors = [ACCENT, "#00ADD8", "#3572A5", "#701516"]
        BAR_LEFT = -3.0
        bars = VGroup()
        for i, (name, s, c) in enumerate(zip(langs, speeds, colors)):
            y = 1.5 - i * 1.0
            label = Text(name, font_size=28).move_to([-4.5, y, 0])
            bar = Rectangle(width=0.01, height=0.5, fill_opacity=1,
                            stroke_width=0, fill_color=c)
            bar.move_to([BAR_LEFT, y, 0], aligned_edge=LEFT)
            target = bar.copy().stretch_to_fit_width(8 * s)
            target.move_to([BAR_LEFT, y, 0], aligned_edge=LEFT)
            bars.add(label, bar, target)
            self.play(FadeIn(label), Transform(bar, target), run_time=0.5)
        self.wait(1.5)
        self.play(FadeOut(*self.mobjects))

        # 0:15–0:40  crate tile rain
        subtitle = Text("But building a web app takes...", font_size=44, color=MUTED)
        self.play(Write(subtitle), run_time=1.0)
        self.wait(0.3)
        self.play(subtitle.animate.to_edge(UP))

        # Row 0 (4 tiles) and row 1 (3 tiles), each row independently centered.
        ROW_Y = [1.0, -1.2]
        positions = {
            "axum":     (-4.5, ROW_Y[0]),
            "tower":    (-1.5, ROW_Y[0]),
            "sea-orm":  ( 1.5, ROW_Y[0]),
            "utoipa":   ( 4.5, ROW_Y[0]),
            "jwt":      (-3.0, ROW_Y[1]),
            "tracing":  ( 0.0, ROW_Y[1]),
            "sqlx-cli": ( 3.0, ROW_Y[1]),
        }
        tiles = {}
        tile_group = VGroup()
        for name, (x, y) in positions.items():
            t = Text(name, font_size=26, color=FG)
            box = SurroundingRectangle(t, color=MUTED, buff=0.15,
                                       stroke_width=2,
                                       fill_color=BG, fill_opacity=1.0)
            tile = VGroup(box, t).move_to([x, y, 0]).shift(UP * 6)
            tiles[name] = tile
            tile_group.add(tile)
        self.play(
            *[t.animate.shift(DOWN * 6) for t in tile_group],
            run_time=2.0, lag_ratio=0.05,
        )

        MARGIN = 0.2

        def h_arrow(src, dst):
            s = src.get_right() + RIGHT * MARGIN
            e = dst.get_left()  + LEFT  * MARGIN
            return Arrow(s, e, color=WARN, stroke_width=3,
                         buff=0.0, tip_length=0.2)

        def elbow_down_arrow(src, dst):
            s = src.get_bottom() + DOWN * MARGIN
            e = dst.get_top()    + UP   * MARGIN
            mid_y = (s[1] + e[1]) / 2
            shaft = VMobject(stroke_color=WARN, stroke_width=3)
            shaft.set_points_as_corners([
                s,
                np.array([s[0], mid_y, 0]),
                np.array([e[0], mid_y, 0]),
                e,
            ])
            tip = ArrowTriangleFilledTip(color=WARN, length=0.2).scale(1.0)
            tip.rotate(-PI / 2 - tip.tip_angle)
            tip.move_to(e)
            return VGroup(shaft, tip)

        horizontal_deps = [("axum", "tower"), ("tower", "sea-orm"), ("sea-orm", "utoipa")]
        vertical_deps   = [("tower", "jwt"), ("sea-orm", "tracing"), ("utoipa", "sqlx-cli")]

        arrows = VGroup()
        for s, d in horizontal_deps:
            arrows.add(h_arrow(tiles[s], tiles[d]))
        for s, d in vertical_deps:
            arrows.add(elbow_down_arrow(tiles[s], tiles[d]))

        self.play(LaggedStart(*[FadeIn(a) for a in arrows], lag_ratio=0.15))
        self.wait(1.5)

        # 0:40–1:10  Django side
        self.play(FadeOut(subtitle, tile_group, arrows))
        left  = Text("Rust", font_size=40, color=RUST).move_to(LEFT * 4 + UP * 2.5)
        right = Text("Django", font_size=40, color=OK).move_to(RIGHT * 4 + UP * 2.5)
        self.play(FadeIn(left), FadeIn(right))

        rust_items   = ["axum?", "sea-orm?", "utoipa?", "jwt?", "tower?", "serde?"]
        django_items = ["Server", "ORM", "OpenAPI Docs", "Auth", "Middleware", "Serializer"]
        for i, (r, d) in enumerate(zip(rust_items, django_items)):
            rt = Text(r, font_size=26, color=MUTED).move_to(LEFT * 4 + UP * (1.5 - i * 0.7))
            dt = Text(d, font_size=26, color=OK).move_to(RIGHT * 4 + UP * (1.5 - i * 0.7))
            self.play(FadeIn(rt), FadeIn(dt), run_time=0.35)
        self.wait(1.5)

        # 1:10–1:30  punch line
        self.play(FadeOut(*self.mobjects))
        punch = VGroup(
            Text("Django: 1 command.", font_size=48, color=OK),
            Text("Rust: 10+ crates.",  font_size=48, color=RUST),
            Text("?", font_size=140, color=ACCENT),
        ).arrange(DOWN, buff=0.4).move_to(ORIGIN)
        self.play(Write(punch[0]))
        self.play(Write(punch[1]))
        self.play(FadeIn(punch[2], scale=0.5))
        self.wait(1.5)

Manim Community v0.20.1

## Scene 2 — Reinhardt intro + polylithic architecture (1:00)

In [3]:
%%manim -ql -v WARNING Scene2Reinhardt

import numpy as np

class Scene2Reinhardt(Scene):
    def construct(self):
        # 1:30–1:45  logo + tagline
        logo = Text("Reinhardt", font_size=88, weight=BOLD, color=ACCENT)
        tag  = Text("🦀 Django's productivity, Rust's performance",
                    font_size=30, color=FG).next_to(logo, DOWN, buff=0.5)
        self.play(Write(logo), run_time=1.0)
        self.play(FadeIn(tag, shift=UP * 0.3), run_time=0.8)
        self.wait(1.0)
        self.play(logo.animate.scale(0.45).to_edge(UP),
                  FadeOut(tag))

        # 1:45–2:10  hub with nodes arranged as top row / bottom row.
        # Every edge is an elbow (no diagonal segment), and both endpoints
        # are offset by MARGIN so the shaft never touches the hub rim or
        # a node side.
        HUB_R = 0.8
        hub = Circle(radius=HUB_R, color=ACCENT, fill_opacity=1.0).set_fill(ACCENT)
        hub.set_z_index(2)
        hub_label = Text("reinhardt", font_size=22, color=BG).move_to(hub)
        hub_label.set_z_index(3)
        self.play(GrowFromCenter(hub), FadeIn(hub_label))

        crates = [
            "reinhardt-core",  "reinhardt-orm",    "reinhardt-di",      "reinhardt-auth",
            "reinhardt-admin", "reinhardt-api",    "reinhardt-graphql", "reinhardt-ws",
        ]
        TOP_Y, BOT_Y = 2.6, -2.6
        COLS = [-4.8, -1.6, 1.6, 4.8]
        layout = [
            (crates[0], COLS[0], TOP_Y),
            (crates[1], COLS[1], TOP_Y),
            (crates[2], COLS[2], TOP_Y),
            (crates[3], COLS[3], TOP_Y),
            (crates[4], COLS[0], BOT_Y),
            (crates[5], COLS[1], BOT_Y),
            (crates[6], COLS[2], BOT_Y),
            (crates[7], COLS[3], BOT_Y),
        ]
        MARGIN = 0.22
        nodes, edges = VGroup(), VGroup()
        for name, x, y in layout:
            t = Text(name, font_size=18, color=FG)
            box = SurroundingRectangle(t, color=MUTED, buff=0.12,
                                       stroke_width=1.5,
                                       fill_color=BG, fill_opacity=1.0)
            node = VGroup(box, t).move_to([x, y, 0])
            node.set_z_index(2)

            is_top = y > 0
            hub_exit = (hub.get_top() + UP * MARGIN) if is_top \
                       else (hub.get_bottom() + DOWN * MARGIN)
            node_entry = (node.get_bottom() + DOWN * MARGIN) if is_top \
                         else (node.get_top() + UP * MARGIN)
            corner = np.array([node.get_center()[0], hub_exit[1], 0])
            shaft = VMobject(stroke_color=MUTED, stroke_width=1.5)
            shaft.set_points_as_corners([hub_exit, corner, node_entry])
            shaft.set_z_index(1)

            nodes.add(node); edges.add(shaft)

        self.play(LaggedStart(*[Create(e) for e in edges], lag_ratio=0.08),
                  LaggedStart(*[FadeIn(n) for n in nodes], lag_ratio=0.08),
                  run_time=2.5)

        # independence pulse
        for n in nodes:
            self.play(n[0].animate.set_color(ACCENT), run_time=0.15)
            self.play(n[0].animate.set_color(MUTED),  run_time=0.15)
        self.wait(0.5)

        # 2:10–2:30  goal strip
        self.play(FadeOut(hub, hub_label, nodes, edges))
        goal = Text("Build a Blog API in 15 minutes",
                    font_size=40, color=ACCENT).shift(UP * 0.8)
        steps = ["Model", "Migration", "API", "Admin", "Auth"]
        strip = VGroup(*[Text(s, font_size=28, color=FG) for s in steps])
        strip.arrange(RIGHT, buff=1.0).next_to(goal, DOWN, buff=0.8)
        arrows_between = VGroup(*[
            Arrow(strip[i].get_right(), strip[i + 1].get_left(),
                  color=MUTED, stroke_width=2, buff=0.1)
            for i in range(len(strip) - 1)
        ])
        self.play(FadeIn(goal))
        self.play(LaggedStart(*[FadeIn(t) for t in strip], lag_ratio=0.15),
                  LaggedStart(*[GrowArrow(a) for a in arrows_between], lag_ratio=0.15))
        self.wait(2.0)

Manim Community v0.20.1

## Scene 10 — Recap, comparison, CTA (1:30)

In [4]:
%%manim -ql -v WARNING Scene10Closing

class Scene10Closing(Scene):
    def construct(self):
        # 13:30–14:10  checklist + line counter
        checklist = VGroup(*[
            Text(f"✅ {label}", font_size=36, color=OK)
            for label in ["Model", "Migration", "CRUD API", "Admin", "Auth + DI"]
        ]).arrange(DOWN, aligned_edge=LEFT, buff=0.35).to_edge(LEFT, buff=1.2)

        counter = Text("≈ 60 lines of Rust",
                       font_size=46, color=ACCENT).to_edge(RIGHT, buff=1.2)

        self.play(LaggedStart(*[FadeIn(item, shift=RIGHT * 0.3) for item in checklist],
                              lag_ratio=0.2))
        self.play(Write(counter))
        self.wait(2.5)

        # 14:10–14:50  CTA
        self.play(FadeOut(checklist, counter))
        cmd = Text("$ cargo install reinhardt-admin-cli",
                   font_size=36, color=FG, font=MONO_FONT)
        repo = Text("github.com/kent8192/reinhardt-web",
                    font_size=28, color=MUTED).next_to(cmd, DOWN, buff=0.5)
        site = Text("reinhardt-web.dev",
                    font_size=28, color=ACCENT).next_to(repo, DOWN, buff=0.3)
        self.play(Write(cmd))
        self.play(FadeIn(repo), FadeIn(site))
        self.wait(2.0)

        # 14:50–15:00  closing tagline
        self.play(FadeOut(cmd, repo, site))
        closer = Text("Django's productivity. Rust's performance.",
                      font_size=40, weight=BOLD, color=ACCENT)
        self.play(FadeIn(closer, scale=0.9))
        self.wait(2.0)
        self.play(FadeOut(closer))

Manim Community v0.20.1

## (Optional) Stitch the three scenes into a single MP4

Uncomment when you have all three renders and want a single concatenated preview. Final editing should happen in a video editor, not here.

In [5]:
# from pathlib import Path
# from moviepy import VideoFileClip, concatenate_videoclips
#
# root = Path("media/videos/manim/480p15")
# clips = [VideoFileClip(str(root / f"{name}.mp4"))
#          for name in ["Scene1Opening", "Scene2Reinhardt", "Scene10Closing"]]
# concatenate_videoclips(clips).write_videofile("media/blog_demo_manim_only.mp4",
#                                               codec="libx264", audio=False)